# Stack rasters and extract rasters and tables with WGS84 coordinates

This script loops over years, loads all spatial data files (NPP, climate yearly, climate vegetation period, elevation, region) for a year, stacks them, reprojects them to WGS84, and extracts a raster and a table for each year. The table contains a column for each raster band, WGS84 coordinates of the MODIS pixel centers, the year, and a row for each pixel that does not have a missing value for NPP. The data is exported as a parquet file.

Overall workflow: Stack MODIS rasters, reproject to WGS84, add administrative regions, and export raster + table

For each year:
1. Stack all MODIS-grid rasters
2. Reproject the stack to WGS84
4. Extract valid NPP pixels as table
5. Assign administrative regions
6. Export the WGS84 stack as a GeoTIFF and the table as Parquet

## 0. Load and inspect data

In [43]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import polars as pl
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.transform import xy
from pyproj import Transformer
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling,
)

In [44]:
# Define years to process
YEARS = range(2003, 2024)

# Define paths to data
# path = Path("/Volumes/TOSHIBA EXT/non-equilibrium/data")
path = Path("/Users/Wanja/Documents/non-equilibrium_data")
climate_path = Path(path, "climate_yearly_new")
npp_path = Path(path, "npp")
elevation_path = Path(path, "elevation/Elevation.tif")
regions_path = Path(path, "world-administrative-boundaries/world-administrative-boundaries.shp")

# Define output paths
raster_modis_output_path = path / "rasters_modis_new"
raster_wgs84_output_path = path / "rasters_wgs84_new"
table_output_path = path / "tables_wgs84_new"

In [45]:
# Inspect raster files 
files = [
    climate_path / "Climate_yearly_2002.tif",
    climate_path / "Climate_vegetation_period_2002.tif",
    npp_path / "NPP_2002-01-01.tif",
    elevation_path,
]

for file in files:
    with rasterio.open(file) as src:
        print("=" * 60)
        print(file.name)
        print("=" * 60)
        print(f"Bands: {src.count}")
        print(f"CRS: {src.crs}")
        print(f"Resolution: {src.res}")
        print(f"Nodata: {src.nodata}")
        print()

        for i, desc in enumerate(src.descriptions, start=1):
            print(f"Band {i}: {desc}")

Climate_yearly_2002.tif
Bands: 7
CRS: PROJCS["MODIS Sinusoidal",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Resolution: (5000.0, 5000.0)
Nodata: nan

Band 1: tmmn_mean
Band 2: tmmx_mean
Band 3: tmmn_min
Band 4: tmmn_max
Band 5: tmmx_min
Band 6: tmmx_max
Band 7: pr_sum
Climate_vegetation_period_2002.tif
Bands: 8
CRS: PROJCS["MODIS Sinusoidal",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_east

In [46]:
npp_file = npp_path / "NPP_2002-01-01.tif"

with rasterio.open(npp_file) as src:

    npp_band2 = src.read(2)
    
    valid = ~np.isnan(npp_band2)

    print("Raster shape:", npp_band2.shape)
    print("Total pixels:", npp_band2.size)
    print("Valid pixels:", valid.sum())
    print("Missing pixels:", (~valid).sum())
    print("Percentage valid:", valid.mean() * 100)

Raster shape: (4004, 8008)
Total pixels: 32064032
Valid pixels: 2913692
Missing pixels: 29150340
Percentage valid: 9.087104204486822


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2440744811.py:5: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  npp_band2 = src.read(2)


In [47]:
# Load regions shapefile
regions = gpd.read_file(regions_path)

regions = regions[[
    "iso3",
    "name",
    "continent",
    "region",
    "geometry"
]]

In [48]:
# Inspect regions GeoDataFrame
print("=" * 60)
print("World administrative boundaries")
print("=" * 60)

print(f"Number of regions: {len(regions)}")
print(f"CRS: {regions.crs}")
print(f"Geometry types:")
print(regions.geometry.geom_type.value_counts())

print("\nColumns:")
print(regions.columns.tolist())

print("\nFirst rows:")
display(regions.head())

World administrative boundaries
Number of regions: 256
CRS: EPSG:4326
Geometry types:
Polygon         141
MultiPolygon    115
Name: count, dtype: int64

Columns:
['iso3', 'name', 'continent', 'region', 'geometry']

First rows:


,iso3,name,continent,region,geometry
0,UGA,Uganda,Africa,Eastern Africa,"POLYGON ((33.9211 -1.00194, 33.92027 -1.00111,..."
1,UZB,Uzbekistan,Asia,Central Asia,"POLYGON ((70.97081 42.25467, 70.98054 42.26205..."
2,IRL,Ireland,Europe,Northern Europe,"MULTIPOLYGON (((-9.97014 54.02083, -9.93833 53..."
3,ERI,Eritrea,Africa,Eastern Africa,"MULTIPOLYGON (((40.13583 15.7525, 40.12861 15...."
4,NaN,Ma'tan al-Sarra,Africa,Northern Africa,"POLYGON ((33.25104 21.99977, 34.15064 21.99603..."


## 0. Helper functions

In [49]:
def get_band_names(src, prefix):
    """
    Return band descriptions, falling back to generic names.

    Parameters
    ----------
    src : rasterio dataset
        Open raster dataset.
    prefix : str
        Prefix used for unnamed bands.

    Returns
    -------
    list of str
        Band names.
    """

    names = []

    for i, description in enumerate(src.descriptions, start=1):

        if description is None:
            description = f"{prefix}_band_{i}"

        names.append(description)

    return names

In [50]:
def generate_windows(src):
    """
    Generate raster block windows for block-wise processing. 
    Windows are generated based on the block shapes of the raster dataset (256x256 pixels).

    Parameters
    ----------
    src : rasterio.io.DatasetReader
        Open raster dataset.

    Yields
    ------
    rasterio.windows.Window
        Window covering one internal raster block.
    """

    block_h, block_w = src.block_shapes[0]

    for row in range(0, src.height, block_h):
        for col in range(0, src.width, block_w):

            yield Window(
                col,
                row,
                min(block_w, src.width-col),
                min(block_h, src.height-row)
            )

In [ ]:
def window_coordinates(src, window):
    """
    Return WGS84 coordinates for the center of each pixel in a raster window.

    Parameters
    ----------
    src : rasterio.io.DatasetReader
        Open raster dataset.
    window : rasterio.windows.Window
        Raster window to process.

    Returns
    -------
    tuple of numpy.ndarray
        Longitude and latitude arrays with shape (height, width).
    """

    rows, cols = np.meshgrid(
        np.arange(
            window.row_off,
            window.row_off + window.height
        ),
        np.arange(
            window.col_off,
            window.col_off + window.width
        ),
        indexing="ij"
    )

    xs, ys = rasterio.transform.xy(
        src.transform,
        rows,
        cols,
        offset="center"
    )

    lon = np.asarray(xs).reshape(
        window.height,
        window.width
    )

    lat = np.asarray(ys).reshape(
        window.height,
        window.width
    )

    return lon, lat

In [52]:
def get_input_files(year):
    """
    Return all raster files required for one year.
    """

    return [
        (
            climate_path / f"Climate_yearly_{year}.tif",
            "climate_yearly",
        ),
        (
            climate_path / f"Climate_vegetation_period_{year}.tif",
            "climate_vegetation_period",
        ),
        (
            npp_path / f"NPP_{year}-01-01.tif",
            "npp",
        ),
        (
            elevation_path,
            "elevation",
        ),
    ]

## 1. Stack all MODIS-grid rasters

In [ ]:
def create_modis_stack(year, output_file):
    """
    Stack all input rasters on the common MODIS grid.

    All input rasters must have identical spatial dimensions,
    CRS, transform, and pixel alignment.

    Parameters
    ----------
    year : int
        Year to process.

    output_file : Path
        Temporary MODIS-grid stack.

    Returns
    -------
    list of str
        Band names.
    """

    input_files = get_input_files(year)

    # --------------------------------------------------------
    # Open all input rasters
    # --------------------------------------------------------

    datasets = [
        rasterio.open(file)
        for file, _ in input_files
    ]

    try:

        reference = datasets[0]

        # ----------------------------------------------------
        # Check that all rasters use the same MODIS grid
        # ----------------------------------------------------

        for src in datasets[1:]:

            if src.crs != reference.crs:
                raise ValueError(
                    f"CRS mismatch: {src.name}"
                )

            if src.width != reference.width:
                raise ValueError(
                    f"Width mismatch: {src.name}"
                )

            if src.height != reference.height:
                raise ValueError(
                    f"Height mismatch: {src.name}"
                )

            if src.transform != reference.transform:
                raise ValueError(
                    f"Transform mismatch: {src.name}"
                )

        # ----------------------------------------------------
        # Collect band names
        # ----------------------------------------------------

        band_names = []

        band_names.extend(
            get_band_names(
                datasets[0],
                "climate_yearly",
            )
        )

        band_names.extend(
            get_band_names(
                datasets[1],
                "climate_vegetation_period",
            )
        )

        band_names.extend(
            get_band_names(
                datasets[2],
                "npp",
            )
        )

        # Only use the first elevation band
        band_names.append("elevation_mean")

        # ----------------------------------------------------
        # Create output profile
        # ----------------------------------------------------

        profile = reference.profile.copy()

        profile.update(
            driver="GTiff",
            count=len(band_names),
            dtype="float32",
            nodata=np.nan,
            compress="deflate",
            predictor=2,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        # ----------------------------------------------------
        # Write stack
        # ----------------------------------------------------

        with rasterio.open(output_file, "w", **profile) as dst:

            output_band = 1

            # -----------------------------------------------
            # Climate yearly
            # -----------------------------------------------

            for i in range(1, datasets[0].count + 1):

                data = datasets[0].read(i)

                dst.write(
                    data.astype("float32"),
                    output_band,
                )

                dst.set_band_description(
                    output_band,
                    band_names[output_band - 1],
                )

                output_band += 1

            # -----------------------------------------------
            # Climate vegetation period
            # -----------------------------------------------

            for i in range(1, datasets[1].count + 1):

                data = datasets[1].read(i)

                dst.write(
                    data.astype("float32"),
                    output_band,
                )

                dst.set_band_description(
                    output_band,
                    band_names[output_band - 1],
                )

                output_band += 1

            # -----------------------------------------------
            # NPP
            # -----------------------------------------------

            for i in range(1, datasets[2].count + 1):

                data = datasets[2].read(i)

                dst.write(
                    data.astype("float32"),
                    output_band,
                )

                dst.set_band_description(
                    output_band,
                    band_names[output_band - 1],
                )

                output_band += 1

            # -----------------------------------------------
            # Elevation
            # -----------------------------------------------

            elevation = datasets[3].read(1).astype("float32")

            # Convert elevation nodata value to NaN
            elevation[elevation == -9999] = np.nan

            if datasets[3].nodata is not None:

                elevation[
                    elevation == datasets[3].nodata
                ] = np.nan

            dst.write(
                elevation,
                output_band,
            )

            dst.set_band_description(
                output_band,
                "elevation_mean",
            )

    finally:

        for src in datasets:
            src.close()

    return band_names

## 2. Reproject MODIS stack to WGS84 and export the raster

In [ ]:
def reproject_to_wgs84(
    input_file,
    output_file,
    band_names,
):
    """
    Reproject a MODIS-grid raster stack to WGS84.

    Resampling is selected separately for each band.

    Parameters
    ----------
    input_file : Path
        MODIS-grid stack.

    output_file : Path
        WGS84 output raster.

    band_names : list of str
        Names of the raster bands.
    """

    with rasterio.open(input_file) as src:

        # ----------------------------------------------------
        # Calculate WGS84 grid
        # ----------------------------------------------------

        transform, width, height = (
            calculate_default_transform(
                src.crs,
                "EPSG:4326",
                src.width,
                src.height,
                *src.bounds,
            )
        )

        profile = src.profile.copy()

        profile.update(
            crs="EPSG:4326",
            transform=transform,
            width=width,
            height=height,
            compress="deflate",
            predictor=2,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        # ----------------------------------------------------
        # Open output
        # ----------------------------------------------------

        with rasterio.open(
            output_file,
            "w",
            **profile,
        ) as dst:

            for band_index, band_name in enumerate(
                band_names,
                start=1,
            ):

                # --------------------------------------------
                # Choose resampling method
                # --------------------------------------------

                # Continuous variables: bilinear interpolation
                resampling = Resampling.bilinear

                # Vegetation length: nearest-neighbour resampling
                if band_name == "vegetation_length":
                    resampling = Resampling.nearest

                # --------------------------------------------
                # Reproject
                # --------------------------------------------

                reproject(
                    source=rasterio.band(
                        src,
                        band_index,
                    ),
                    destination=rasterio.band(
                        dst,
                        band_index,
                    ),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=transform,
                    dst_crs="EPSG:4326",
                    dst_nodata=np.nan,
                    resampling=resampling,
                )

                dst.set_band_description(
                    band_index,
                    band_name,
                )


## 3. Extract table from WGS84 raster

In [ ]:
def process_wgs84_block(
    src,
    window,
    year,
):
    """
    Extract valid NPP pixels from a WGS84 raster block as table.

    Coordinates are calculated directly from the WGS84
    raster transform.
    """

    data = src.read(window=window)

    # --------------------------------------------------------
    # Identify NPP band
    # --------------------------------------------------------

    npp_band = None

    for i, name in enumerate(src.descriptions):

        if name == "Npp":
            npp_band = i
            break

    if npp_band is None:
        raise ValueError(
            "Band 'Npp' was not found in the raster."
        )

    # --------------------------------------------------------
    # Valid NPP pixels
    # --------------------------------------------------------

    npp = data[npp_band]

    mask = ~np.isnan(npp)

    if mask.sum() == 0:
            return None

    # --------------------------------------------------------
    # Coordinates
    # --------------------------------------------------------

    lon, lat = window_coordinates(
        src,
        window
    )

    # --------------------------------------------------------
    # Create output dictionary
    # --------------------------------------------------------

    output = {}

    for i, name in enumerate(
        src.descriptions
    ):

        if name is None:

            name = f"band_{i + 1}"

        output[name] = data[i][mask]

    output["longitude"] = lon[mask]
    output["latitude"] = lat[mask]
    output["year"] = np.full(mask.sum(), year, dtype=np.int16)

    return pl.DataFrame(output)

## 4. Add administrative regions

In [ ]:
def add_regions(df):
    """
    Assign administrative regions to pixel coordinates using a spatial join. 
    If one pixel falls within multiple regions, only the first match is kept.

    Parameters
    ----------
    df : polars.DataFrame
        Table containing longitude and latitude columns in WGS84.

    Returns
    -------
    polars.DataFrame
        Input table with administrative region attributes appended.
    """

    gdf = gpd.GeoDataFrame(
        df.to_pandas(),
        geometry=gpd.points_from_xy(
            df["longitude"],
            df["latitude"]
        ),
        crs="EPSG:4326"
    )
    
    joined = gpd.sjoin(
        gdf,
        regions,
        predicate="within",
        how="left"
    )
    
    # A point should normally belong to only one region.
    # Keep the first match in case of overlapping polygons.
    joined = (
        joined
        .drop_duplicates(
            subset=["longitude", "latitude"]
        )
    )

    joined = joined.drop(columns=["geometry", "index_right"])

    return pl.from_pandas(joined)

## 5. Process one year and export the table

In [ ]:
def process_year(year):
    """
    Create the MODIS stack, reproject it to WGS84,
    export the raster, and extract the pixel table.
    """

    print("=" * 60)
    print(f"Processing {year}")
    print("=" * 60)

    # --------------------------------------------------------
    # File paths
    # --------------------------------------------------------

    modis_stack = (
        raster_modis_output_path
        / f"stack_modis_{year}.tif"
    )

    wgs84_raster = (
        raster_wgs84_output_path
        / f"stack_wgs84_{year}.tif"
    )

    table_file = (
        table_output_path
        / f"table_wgs84_{year}.parquet"
    )

    # --------------------------------------------------------
    # 1. Stack MODIS rasters
    # --------------------------------------------------------

    print("1/4 Creating MODIS stack...")

    band_names = create_modis_stack(
        year,
        modis_stack,
    )

    # --------------------------------------------------------
    # 2. Reproject to WGS84
    # --------------------------------------------------------

    print("2/4 Reprojecting to WGS84...")

    reproject_to_wgs84(
        modis_stack,
        wgs84_raster,
        band_names,
    )

    # --------------------------------------------------------
    # 3. Extract table
    # --------------------------------------------------------

    print("3/4 Extracting pixel table...")

    tables = []

    with rasterio.open(wgs84_raster) as src:

        for window in generate_windows(src):

            df = process_wgs84_block(
                src,
                window,
                year,
            )

            if df is None:
                continue

            # Add administrative information
            df = add_regions(df)

            tables.append(df)

    # --------------------------------------------------------
    # Combine and save
    # --------------------------------------------------------

    if not tables:

        print("No valid pixels found.")
        return

    yearly_table = pl.concat(tables)

    # --------------------------------------------------------
    # 4. Export table
    # --------------------------------------------------------

    print("4/4 Writing Parquet...")

    yearly_table.write_parquet(
        table_file
    )

    print()
    print(f"Raster: {wgs84_raster}")
    print(f"Table:  {table_file}")
    print(f"Rows:   {yearly_table.height}")

## 6. Main

In [58]:
for year in YEARS:

    print(year)

    process_year(year)

2003
Processing 2003
1/4 Creating MODIS stack...


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:128: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[0].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:148: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[1].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:168: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[2].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:186: DeprecationWarning: Setting the shape

2/4 Reprojecting to WGS84...
3/4 Extracting pixel table...
4/4 Writing Parquet...

Raster: /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2003.tif
Table:  /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2003.parquet
Rows:   4218230
2004
Processing 2004
1/4 Creating MODIS stack...


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:128: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[0].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:148: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[1].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:168: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[2].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:186: DeprecationWarning: Setting the shape

2/4 Reprojecting to WGS84...
3/4 Extracting pixel table...
4/4 Writing Parquet...

Raster: /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2004.tif
Table:  /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2004.parquet
Rows:   4218230
2005
Processing 2005
1/4 Creating MODIS stack...


...

2/4 Reprojecting to WGS84...
3/4 Extracting pixel table...
4/4 Writing Parquet...

Raster: /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2021.tif
Table:  /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2021.parquet
Rows:   4218230
2022
Processing 2022
1/4 Creating MODIS stack...


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:128: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[0].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:148: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[1].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:168: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[2].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:186: DeprecationWarning: Setting the shape

2/4 Reprojecting to WGS84...
3/4 Extracting pixel table...
4/4 Writing Parquet...

Raster: /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2022.tif
Table:  /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2022.parquet
Rows:   4218230
2023
Processing 2023
1/4 Creating MODIS stack...


/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:128: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[0].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:148: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[1].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:168: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = datasets[2].read(i)
/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_63248/2325306055.py:186: DeprecationWarning: Setting the shape

2/4 Reprojecting to WGS84...
3/4 Extracting pixel table...
4/4 Writing Parquet...

Raster: /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2023.tif
Table:  /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2023.parquet
Rows:   4218230


# TODO:

- commit git
- backup data on hard drive